In [ ]:
from dotenv import load_dotenv
import os
from langchain_google_genai import ChatGoogleGenerativeAI
from exa_py import Exa
from langchain_core.tools import tool
from langchain.agents import create_tool_calling_agent, AgentExecutor
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

# Load environment variables
load_dotenv()
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
EXA_API_KEY = os.getenv("EXA_API_KEY")

# Initialize Exa
exa = Exa(api_key=os.environ["EXA_API_KEY"])

print(GEMINI_API_KEY)
print(EXA_API_KEY)


In [ ]:
# Define Exa tools
@tool
def search_and_contents(query: str):
    """Search for webpages based on the query and retrieve their contents."""
    response = exa.search_and_contents(
        query, use_autoprompt=True, num_results=5, text=True, highlights=True
    )
    # Format response as a string
    formatted_results = "\n".join([
        f"Title: {result.title}\nURL: {result.url}\nText: {result.text}\nHighlights: {result.highlights}"
        for result in response.results
    ])
    return formatted_results or "No results found."

@tool
def find_similar_and_contents(url: str):
    """Search for webpages similar to a given URL and retrieve their contents.
    The url passed in should be a URL returned from `search_and_contents`.
    """
    try:
        response = exa.find_similar_and_contents(url, num_results=5, text=True, highlights=True)
        # Format response as a string
        formatted_results = "\n".join([
            f"Title: {result.title}\nURL: {result.url}\nText: {result.text}\nHighlights: {result.highlights}"
            for result in response.results
        ])
        return formatted_results or "No similar results found."
    except Exception as e:
        return f"Error finding similar content: {str(e)}"

tools = [search_and_contents, find_similar_and_contents]

In [ ]:


# Initialize Gemini 2.0 Flash
llm = ChatGoogleGenerativeAI(
    model="gemini-2.0-flash",
    google_api_key=GEMINI_API_KEY,
    temperature=0.3,
    max_output_tokens=16000
)
# Define prompt template
prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a helpful assistant that uses Exa to search the web and find similar content. 
    - Use `search_and_contents` for general queries to find webpages and their contents.
    - Use `find_similar_and_contents` only when asked to find content similar to a specific URL.
    - Summarize results concisely in a ranked list if applicable, or provide a clear answer based on the retrieved information.
    - If no relevant results are found, state so clearly."""),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad")
])

# Bind tools to LLM
llm_with_tools = llm.bind_tools(tools)

# Create agent
agent = create_tool_calling_agent(llm=llm_with_tools, tools=tools, prompt=prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

query = "give me best image generation models"

try:
    response = agent_executor.invoke({"input": query})
    print("Agent output:\n", response["output"])
except Exception as e:
    print("Agent execution error:", str(e))
